# Feature Store with Feast and Neo4j

This notebook builds a feature store using [Feast](https://feast.dev/) with Neo4j Desktop as the feature computation engine and SQLite as the online store. Five standard e-commerce features are computed with Cypher. A sixth, `community_id`, is a graph-native feature: customers with **similar purchase mixes** (cosine similarity over their order counts per category, using GDS node similarity) are grouped with Louvain community detection from the Graph Data Science (GDS) library. The notebook then tests whether that feature tells a model anything that `favorite_category` doesn't.

Features are computed at several points in time, so Feast can serve point-in-time correct training data as well as online features.

### Before you start

- Open Neo4j Desktop and ensure you have a project running.
- The Graph Data Science (GDS) plugin needs to be installed.
    - Select the **three dots (...)** in the top-right.
    - Select **Plugins**.
    - Install **Graph Data Science Library**.

> **Warning:** the data-loading step removes **all data, indexes and constraints** from the database before loading. Use a database you are happy to wipe.

### Before running the notebook

Export the following in your shell:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="your-database"
```

## 1. Install Dependencies

In [1]:
%pip install feast==0.66.0 \
             neo4j==6.2.0 \
             pandas==2.3.3 \
             pyarrow==25.0.1 \
             tabulate==0.10.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import numpy as np
import pandas as pd
import random
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from math import comb
from feast import FeatureStore
from neo4j import GraphDatabase
from tabulate import tabulate

## 3. Configuration

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

SEED = 42

# GDS in-memory graph names
BIPARTITE_GRAPH  = "customer-category"
SIMILARITY_GRAPH = "customer-similarity"

# Synthetic data: each customer belongs to a hidden segment defined by two categories
CATEGORIES = ["Electronics", "Clothing", "Home", "Books", "Sports"]
SEGMENTS = [
    ("Electronics", "Home"),
    ("Electronics", "Sports"),
    ("Clothing",    "Books"),
    ("Clothing",    "Home"),
    ("Books",       "Sports"),
]
PRIMARY_SHARE   = 0.55   # share of orders in the segment's primary category
SECONDARY_SHARE = 0.30   # share in its secondary category (the remaining 15% are drawn from all categories)

# Customer similarity graph: keep each customer's K most similar customers above the cutoff
SIMILARITY_TOP_K  = 15
SIMILARITY_CUTOFF = 0.8

print("Credentials set.")

Credentials set.


## 4. Connect to Neo4j

In [4]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity()) # None is expected
print("Neo4j connection verified.")

None
Neo4j connection verified.


In [5]:
# Record the versions this notebook is running against
with driver.session(database = NEO4J_DATABASE) as session:
    for r in session.run("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition"):
        print(f"{r['name']}: {r['versions'][0]} ({r['edition']})")
    try:
        print("GDS:", session.run("RETURN gds.version() AS v").single()["v"])
    except Exception as e:
        print("GDS not available:", str(e).splitlines()[0][:160])

Neo4j Kernel: 2026.07.0 (enterprise)
Cypher: 5 ()
GDS: 2026.07.0


> We leave server notifications on, because they are worth reading. One known warning: GDS reports that the `schema` field returned by `gds.graph.drop` is deprecated. Every `gds.graph.drop` call in this notebook therefore yields only `graphName`, which avoids that warning.

## 5. Create Feature Definitions

We write the files that Feast needs:

- `feature_store.yaml` -- project-level config, pointing at SQLite for the online store
- `features/customer_features.py` -- the entity and feature view declarations

The schema includes five standard e-commerce aggregations plus `community_id`, the graph-native feature we'll compute with GDS.

Because this version of the notebook changes the data and the schema of the offline store, we start from a clean `feature_repo` folder.

> SQLite (online) and Parquet (offline) are convenient for a local demo. For production, Feast supports stores built for scale and concurrency, such as Redis or DynamoDB for online serving and a warehouse for offline data.

In [6]:
# Start clean so that stale registry/online-store files from earlier runs can't interfere
shutil.rmtree("feature_repo", ignore_errors = True)

os.makedirs("feature_repo/data",     exist_ok = True)
os.makedirs("feature_repo/features", exist_ok = True)

feature_store_yaml = """
project: ecommerce_features
registry: feature_repo/registry.db
provider: local

online_store:
    type: sqlite
    path: feature_repo/online_store.db
"""

with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("Feature store configuration created.")

Feature store configuration created.


In [7]:
customer_features_py = """
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String, ValueType
from datetime import timedelta

customer = Entity(
    name = "customer_id",
    description = "Customer identifier",
    value_type = ValueType.STRING
)

customer_features = FeatureView(
    name = "customer_features",
    entities = [customer],
    ttl = timedelta(days = 90),
    schema = [
        Field(name = "total_orders",          dtype = Int64),
        Field(name = "total_spent",           dtype = Float32),
        Field(name = "avg_order_value",       dtype = Float32),
        Field(name = "days_since_last_order", dtype = Int64),
        Field(name = "favorite_category",     dtype = String),
        Field(name = "community_id",          dtype = Int64),
    ],
    online = True,
    source = FileSource(
        path = "data/customer_features.parquet",
        timestamp_field = "event_timestamp"
    ),
)
"""

with open("feature_repo/features/customer_features.py", "w") as f:
    f.write(customer_features_py)

print("Feature definitions created.")

Feature definitions created.


## 6. Load Sample Data into Neo4j

We first clear the database (all constraints, indexes and data), then generate 100 customers, 3,000 orders and 5 product categories over the last 90 days.

Each customer belongs to a **hidden segment**, defined by a primary and a secondary category: 55% of the segment's orders are in its primary category, 30% in its secondary category and the remaining 15% are drawn uniformly from all five categories. There are five segments:

| Segment | Primary | Secondary |
|---|---|---|
| 1 | Electronics | Home |
| 2 | Electronics | Sports |
| 3 | Clothing | Books |
| 4 | Clothing | Home |
| 5 | Books | Sports |

The design is deliberate. Two segments share Electronics as their primary category and two share Clothing, so **`favorite_category` cannot tell those segments apart**. Only the combination of categories can. That gives the graph feature something a single aggregation doesn't capture.

**The segment is stored on the `Customer` node for evaluation only. It is never used to compute any feature.**

To make the churn demo meaningful, we also pick some customers who have gone quiet:

- 8 *churned* customers whose most recent order is more than 60 days old
- 10 *cooling* customers whose most recent order is more than 30 days old

The graph model is `(Customer)-[:PLACED]->(Order)-[:IN_CATEGORY]->(Category)`. Timestamps are timezone-aware (UTC) throughout.

In [8]:
def clear_database(session):
    # Drop all constraints
    constraints = session.run("SHOW CONSTRAINTS YIELD name RETURN name").data()
    for c in constraints:
        session.run(f"DROP CONSTRAINT {c['name']} IF EXISTS")

    # Drop all non-lookup indexes
    indexes = session.run("""
        SHOW INDEXES YIELD name, type
        WHERE type <> 'LOOKUP'
        RETURN name
    """).data()
    for idx in indexes:
        session.run(f"DROP INDEX {idx['name']} IF EXISTS")

    # Delete all nodes and relationships in batches
    while True:
        result = session.run("""
            MATCH (n) WITH n LIMIT 1000
            DETACH DELETE n
            RETURN count(*) AS deleted
        """)
        if result.single()["deleted"] == 0:
            break

    print("Database cleared -- constraints, indexes and data removed.")

def generate_orders():
    """Generate synthetic orders. Returns (orders, churned_customers, cooling_customers)."""

    random.seed(SEED)

    customers  = [f"CUST_{i:04d}" for i in range(1, 101)]

    # Hidden segment: (primary category, secondary category)
    segment_of = {cid: random.choice(SEGMENTS) for cid in customers}

    # Customers who have gone quiet, so the churn demo has something to find
    quiet   = random.sample(customers, 18)
    churned = sorted(quiet[:8])
    cooling = sorted(quiet[8:])

    now = datetime.now(timezone.utc)

    orders = []
    for i in range(3000):
        cid                  = random.choice(customers)
        primary, secondary   = segment_of[cid]
        r                    = random.random()

        if r < PRIMARY_SHARE:
            category = primary
        elif r < PRIMARY_SHARE + SECONDARY_SHARE:
            category = secondary
        else:
            category = random.choice(CATEGORIES)

        if cid in churned:
            days_ago = random.randint(65, 90)
        elif cid in cooling:
            days_ago = random.randint(32, 90)
        else:
            days_ago = random.randint(0, 90)

        orders.append({
            "customer_id":      cid,
            "segment":          f"{primary} + {secondary}",
            "order_id":         f"ORD_{i:06d}",
            "order_timestamp":  (now - timedelta(
                                    days  = days_ago,
                                    hours = random.randint(0, 23)
                                )).isoformat(),
            "order_value":      round(random.uniform(10, 500), 2),
            "product_category": category,
            "quantity":         random.randint(1, 5),
        })

    return orders, churned, cooling

def setup_graph():
    """Clear any existing data and load a fresh customer/order graph."""

    orders_data, churned, cooling = generate_orders()

    with driver.session(database = NEO4J_DATABASE) as session:
        clear_database(session)

        session.run("""
            UNWIND $orders AS o
            MERGE (c:Customer {id: o.customer_id})
              ON CREATE SET c.segment = o.segment
            MERGE (cat:Category {name: o.product_category})
            CREATE (ord:Order {
                id:        o.order_id,
                value:     o.order_value,
                timestamp: datetime(o.order_timestamp),
                quantity:  o.quantity
            })
            MERGE (c)-[:PLACED]->(ord)
            MERGE (ord)-[:IN_CATEGORY]->(cat)
        """, orders = orders_data).consume()

    print(f"Loaded {len(orders_data)} orders for {len({o['customer_id'] for o in orders_data})} customers.")
    print(f"Churned customers (>60 days quiet): {churned}")
    print(f"Cooling customers (>30 days quiet): {cooling}")

    return churned, cooling

churned_customers, cooling_customers = setup_graph()

Database cleared -- constraints, indexes and data removed.
Loaded 3000 orders for 100 customers.
Churned customers (>60 days quiet): ['CUST_0032', 'CUST_0034', 'CUST_0052', 'CUST_0055', 'CUST_0069', 'CUST_0072', 'CUST_0075', 'CUST_0096']
Cooling customers (>30 days quiet): ['CUST_0007', 'CUST_0012', 'CUST_0015', 'CUST_0018', 'CUST_0020', 'CUST_0029', 'CUST_0047', 'CUST_0064', 'CUST_0066', 'CUST_0097']


## 7. Helper Functions

**Building the community feature takes four steps**, each a small function below:

1. `project_graph` builds a GDS projection of customers and categories **as of a given date**, with each customer-category relationship weighted by the number of orders. It uses the current Cypher projection (the `gds.graph.project` aggregation function), which replaces the deprecated `gds.graph.project.cypher` procedure.
2. `similar_customers` runs GDS **node similarity** (cosine, weighted by order count) on that graph, so customers whose category mixes point in the same direction score close to 1.
3. `project_similarity_graph` turns those pairs into a customer-to-customer graph, weighted by similarity.
4. `stream_louvain` runs **Louvain community detection** on it.

Why not run Louvain directly on the customer-category graph? Two segments that share a primary category are both attached to the same category node, so Louvain tends to pull them into one community. Section 10 measures this. The similarity graph compares whole category mixes instead, which is what separates those segments.

Other helpers:

- `align_communities` keeps community IDs stable across snapshots. Louvain labels are arbitrary and can change from run to run, which would break a model trained on earlier IDs. We relabel each snapshot to match the previous one by greatest membership overlap.
- `adjusted_rand_index` measures how closely two groupings agree (1.0 = identical, around 0 = chance), so we can check them against the hidden segments.
- `heldout_losses` scores how well a grouping predicts the category of **future** orders (section 11).

In [9]:
def drop_graph(session, name):
    """Drop a GDS projection if it exists. We yield only graphName, because the other
    returned fields include `schema`, which GDS reports as deprecated."""
    session.run("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName", name = name).consume()

def project_graph(session, name, as_of_iso, undirected = False):
    """Project customers and categories from orders placed up to `as_of_iso`.
    Each customer-category relationship is weighted by order count.
    By default relationships point from customer to category, which is what node
    similarity needs; pass undirected = True for community detection on this graph."""

    drop_graph(session, name)

    config = "{undirectedRelationshipTypes: ['*']}" if undirected else "{}"

    return session.run("""
        MATCH (c:Customer)-[:PLACED]->(o:Order)-[:IN_CATEGORY]->(cat:Category)
        WHERE o.timestamp <= datetime($as_of)
        WITH c, cat, count(o) AS order_count
        WITH gds.graph.project(
            $name, c, cat,
            {
                sourceNodeLabels:       labels(c),
                targetNodeLabels:       labels(cat),
                relationshipType:       'PURCHASED_IN',
                relationshipProperties: {order_count: order_count}
            },
            __CONFIG__
        ) AS g
        RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS relationships
    """.replace("__CONFIG__", config), name = name, as_of = as_of_iso).single()

def collapse_pairs(df):
    """Node similarity can report a pair in both directions. Keep one row per
    unordered pair, with the higher score."""
    out = df.copy()
    out["a"] = np.where(out["src"] < out["dst"], out["src"], out["dst"])
    out["b"] = np.where(out["src"] < out["dst"], out["dst"], out["src"])
    return (out.groupby(["a", "b"], as_index = False)["similarity"].max()
               .rename(columns = {"a": "src", "b": "dst"}))

def similar_customers(session, name):
    """Cosine similarity between customers' category mixes (GDS node similarity)."""

    config = {
        "similarityMetric":           "COSINE",
        "relationshipWeightProperty": "order_count",
        "topK":                       SIMILARITY_TOP_K,
        "similarityCutoff":           SIMILARITY_CUTOFF,
    }

    result = session.run("""
        CALL gds.nodeSimilarity.stream($name, $config)
        YIELD node1, node2, similarity
        RETURN gds.util.asNode(node1).id AS src,
               gds.util.asNode(node2).id AS dst,
               similarity
    """, name = name, config = config)

    pairs = pd.DataFrame([r.data() for r in result])

    if pairs.empty:
        raise ValueError("Node similarity returned no pairs -- try lowering SIMILARITY_CUTOFF.")

    return collapse_pairs(pairs)

def project_similarity_graph(session, name, pairs):
    """Project an undirected customer-to-customer graph weighted by similarity."""

    drop_graph(session, name)

    return session.run("""
        UNWIND $pairs AS p
        MATCH (a:Customer {id: p.src}), (b:Customer {id: p.dst})
        WITH gds.graph.project(
            $name, a, b,
            {
                sourceNodeLabels:       labels(a),
                targetNodeLabels:       labels(b),
                relationshipType:       'SIMILAR',
                relationshipProperties: {similarity: p.similarity}
            },
            {undirectedRelationshipTypes: ['*']}
        ) AS g
        RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS relationships
    """, name = name, pairs = pairs.to_dict("records")).single()

def stream_louvain(session, name, weight_property = None):
    """Run Louvain on a projection and return a customer_id -> community_id DataFrame."""

    config = {"consecutiveIds": True}
    if weight_property:
        config["relationshipWeightProperty"] = weight_property

    result = session.run("""
        CALL gds.louvain.stream($name, $config)
        YIELD nodeId, communityId
        WITH gds.util.asNode(nodeId) AS n, communityId
        WHERE n:Customer
        RETURN n.id AS customer_id, communityId AS community_id
    """, name = name, config = config)

    return pd.DataFrame([r.data() for r in result], columns = ["customer_id", "community_id"])

def fill_isolates(df):
    """A customer with no similar neighbour is not in the similarity graph.
    Give each such customer a community of its own so that every customer has an ID."""
    missing = df["community_id"].isna()
    if missing.any():
        start = 0 if missing.all() else int(df["community_id"].max()) + 1
        df.loc[missing, "community_id"] = range(start, start + int(missing.sum()))
    return df

def fetch_segments():
    """The hidden segment of each customer, used only to evaluate results."""
    with driver.session(database = NEO4J_DATABASE) as session:
        rows = session.run("MATCH (c:Customer) RETURN c.id AS customer_id, c.segment AS segment")
        return pd.DataFrame([r.data() for r in rows])

def fetch_orders():
    """Every order as (customer_id, category, ts) with ts in epoch seconds."""
    with driver.session(database = NEO4J_DATABASE) as session:
        rows = session.run("""
            MATCH (c:Customer)-[:PLACED]->(o:Order)-[:IN_CATEGORY]->(cat:Category)
            RETURN c.id AS customer_id, cat.name AS category, o.timestamp.epochSeconds AS ts
        """)
        return pd.DataFrame([r.data() for r in rows])

def align_communities(current, reference):
    """Relabel `current` community IDs to match `reference` by greatest member overlap."""

    merged = current[["customer_id", "community_id"]].merge(
        reference[["customer_id", "community_id"]],
        on = "customer_id", suffixes = ("_cur", "_ref")
    )
    overlap = (merged.groupby(["community_id_cur", "community_id_ref"])
                     .size().reset_index(name = "n")
                     .sort_values("n", ascending = False))

    mapping, used = {}, set()
    for _, r in overlap.iterrows():
        cur, ref = int(r["community_id_cur"]), int(r["community_id_ref"])
        if cur not in mapping and ref not in used:
            mapping[cur] = ref
            used.add(ref)

    # Any community with no counterpart in the reference gets a fresh ID
    next_id = int(max(reference["community_id"].max(), max(mapping.values(), default = -1))) + 1
    for cur in current["community_id"].unique():
        if int(cur) not in mapping:
            mapping[int(cur)] = next_id
            next_id += 1

    aligned = current.copy()
    aligned["community_id"] = aligned["community_id"].map(lambda x: mapping[int(x)]).astype("int64")
    return aligned

def label_agreement(a, b):
    """Fraction of customers (present in both) with the same community_id."""
    m = a[["customer_id", "community_id"]].merge(
        b[["customer_id", "community_id"]], on = "customer_id", suffixes = ("_a", "_b")
    )
    return (m["community_id_a"] == m["community_id_b"]).mean()

def adjusted_rand_index(labels_a, labels_b):
    """Adjusted Rand Index between two labelings (1.0 = identical groupings)."""
    table   = pd.crosstab(pd.Series(list(labels_a)), pd.Series(list(labels_b))).values
    sum_ij  = sum(comb(int(n), 2) for n in table.flatten())
    sum_a   = sum(comb(int(n), 2) for n in table.sum(axis = 1))
    sum_b   = sum(comb(int(n), 2) for n in table.sum(axis = 0))
    total   = comb(int(table.sum()), 2)
    expected = sum_a * sum_b / total
    maximum  = (sum_a + sum_b) / 2
    return 1.0 if maximum == expected else (sum_ij - expected) / (maximum - expected)

def heldout_losses(orders, snapshot, cutoff_ts, group_col, alpha = 1.0):
    """How well does a grouping of customers predict the category of FUTURE orders?

    For each group, estimate the category mix from orders up to `cutoff_ts`
    (with add-`alpha` smoothing). Then score each later order by the negative log
    of the probability that mix gave its category. Lower is better.
    Returns one loss per held-out order."""

    groups = snapshot[["customer_id", group_col]].rename(columns = {group_col: "group"})
    train  = orders[orders["ts"] <= cutoff_ts].merge(groups, on = "customer_id")
    test   = orders[orders["ts"] >  cutoff_ts].merge(groups, on = "customer_id")

    counts = (train.groupby(["group", "category"]).size().unstack(fill_value = 0)
                   .reindex(columns = CATEGORIES, fill_value = 0) + alpha)
    probs  = counts.div(counts.sum(axis = 1), axis = 0)

    idx = pd.MultiIndex.from_arrays([test["group"], test["category"]])
    p   = probs.stack().reindex(idx).fillna(1 / len(CATEGORIES)).values

    return -np.log(p)

print("Helpers defined.")

Helpers defined.


## 8. Compute Features as of a Point in Time

`compute_features_as_of` computes all six features using only orders placed up to a given date:

1. A main Cypher aggregation: counts, totals, averages and recency (measured **relative to the as-of date**)
2. A Cypher query for the favorite category (ties are broken alphabetically so results are deterministic)
3. The community feature: customer-category projection, node similarity, similarity projection, then Louvain

The `event_timestamp` is the as-of date, i.e. the time the features describe, rather than the time of the customer's last order.

In [10]:
STANDARD_QUERY = """
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WHERE o.timestamp <= datetime($as_of)
    RETURN
        c.id                                                    AS customer_id,
        count(o)                                                AS total_orders,
        sum(o.value)                                            AS total_spent,
        avg(o.value)                                            AS avg_order_value,
        duration.inDays(max(o.timestamp), datetime($as_of)).days AS days_since_last_order
"""

FAVORITE_QUERY = """
    MATCH (c:Customer)-[:PLACED]->(o:Order)-[:IN_CATEGORY]->(cat:Category)
    WHERE o.timestamp <= datetime($as_of)
    WITH c.id AS customer_id, cat.name AS category, count(*) AS cnt
    ORDER BY customer_id, cnt DESC, category
    WITH customer_id, collect(category)[0] AS favorite_category
    RETURN customer_id, favorite_category
"""

def detect_communities(session, as_of_iso):
    """Group customers with similar purchase mixes: node similarity, then Louvain."""

    # 1. Customer -> Category graph, weighted by order count
    project_graph(session, BIPARTITE_GRAPH, as_of_iso)

    # 2. Cosine similarity between customers' category mixes
    pairs = similar_customers(session, BIPARTITE_GRAPH)
    drop_graph(session, BIPARTITE_GRAPH)

    # 3. Customer-to-customer graph weighted by similarity, then Louvain
    stats        = project_similarity_graph(session, SIMILARITY_GRAPH, pairs)
    community_df = stream_louvain(session, SIMILARITY_GRAPH, weight_property = "similarity")
    drop_graph(session, SIMILARITY_GRAPH)

    return community_df, {"pairs": len(pairs), "nodes": stats["nodes"], "relationships": stats["relationships"]}

def compute_features_as_of(as_of):
    """Compute all six features for every customer using orders up to `as_of`."""

    as_of_iso = as_of.isoformat()

    with driver.session(database = NEO4J_DATABASE) as session:

        df     = pd.DataFrame([r.data() for r in session.run(STANDARD_QUERY, as_of = as_of_iso)])
        cat_df = pd.DataFrame([r.data() for r in session.run(FAVORITE_QUERY, as_of = as_of_iso)])

        community_df, stats = detect_communities(session, as_of_iso)

    df = df.merge(cat_df, on = "customer_id").merge(community_df, on = "customer_id", how = "left")
    df = fill_isolates(df)

    df["event_timestamp"] = pd.Timestamp(as_of)
    df = df.astype({
        "total_orders":          "int64",
        "days_since_last_order": "int64",
        "community_id":          "int64",
        "total_spent":           "float64",
        "avg_order_value":       "float64",
    })

    return df, stats

print("compute_features_as_of defined.")

compute_features_as_of defined.


## 9. Build Feature Snapshots

We compute features at three points in time: 60 days ago, 30 days ago and now. Each snapshot becomes a set of rows in the offline store, stamped with its as-of date.

Louvain is run independently for each snapshot, so to keep `community_id` consistent over time we align each snapshot to the previous one and we print how much the raw labels would have disagreed without alignment.

For each snapshot we also print the Adjusted Rand Index against the hidden segments. Earlier snapshots have fewer orders per customer, so we expect recovery to improve as data accumulates.

In [11]:
segments_df = fetch_segments()

now = datetime.now(timezone.utc)
SNAPSHOT_OFFSETS_DAYS = [60, 30, 0]

frames, reference = [], None

for days in SNAPSHOT_OFFSETS_DAYS:
    as_of = now - timedelta(days = days)
    df, stats = compute_features_as_of(as_of)

    seg   = df.merge(segments_df, on = "customer_id")
    score = adjusted_rand_index(seg["community_id"], seg["segment"])

    msg = (f"As of {as_of:%Y-%m-%d} ({days:>2} days ago): {len(df)} customers, "
           f"{stats['pairs']} similar pairs, "
           f"{df['community_id'].nunique()} communities, "
           f"ARI vs hidden segments {score:.2f}")

    if reference is not None:
        raw_agreement = label_agreement(df, reference)
        df            = align_communities(df, reference)
        msg += (f"\n    label agreement with previous snapshot: "
                f"raw {raw_agreement:.0%} -> aligned {label_agreement(df, reference):.0%}")

    print(msg)

    reference = df[["customer_id", "community_id"]]
    frames.append(df)

features_df = pd.concat(frames, ignore_index = True)
latest_df   = frames[-1]

print()
print(f"Feature rows: {len(features_df)} ({len(frames)} snapshots).")
print()
print("Latest snapshot (sample):")
sample = latest_df.drop(columns = ["event_timestamp"]).head(4)
print(tabulate(sample, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

As of 2026-08-27 (60 days ago): 100 customers, 844 similar pairs, 5 communities, ARI vs hidden segments 0.95
As of 2026-09-26 (30 days ago): 100 customers, 847 similar pairs, 5 communities, ARI vs hidden segments 0.95
    label agreement with previous snapshot: raw 65% -> aligned 96%
As of 2026-10-26 ( 0 days ago): 100 customers, 856 similar pairs, 5 communities, ARI vs hidden segments 0.98
    label agreement with previous snapshot: raw 63% -> aligned 99%

Feature rows: 300 (3 snapshots).

Latest snapshot (sample):
customer_id      total_orders    total_spent    avg_order_value    days_since_last_order  favorite_category      community_id
-------------  --------------  -------------  -----------------  -----------------------  -------------------  --------------
CUST_0081                  28        7765.10             277.32                        5  Books                             4
CUST_0049                  33        7844.00             237.70                        1  Electronic

## 10. Which Way of Grouping Customers Recovers the Hidden Segments?

Four groupings of the same customers, each scored with the Adjusted Rand Index (ARI) against the hidden segments (1.0 = identical groupings, near 0 = chance):

1. **`favorite_category`**: a plain aggregation, one category per customer.
2. **Louvain on the customer-category graph, unweighted**: every customer is linked to every category equally.
3. **Louvain on the customer-category graph, weighted by order count**. This is the obvious graph approach.
4. **Louvain on the customer-similarity graph**: the method used for `community_id`.

Segments that share a primary category are the hard case. Methods 1 and 3 should struggle with them, because they only see which categories a customer is attached to, not the combination.

In [12]:
as_of_latest = now.isoformat()

with driver.session(database = NEO4J_DATABASE) as session:
    project_graph(session, BIPARTITE_GRAPH, as_of_latest, undirected = True)
    bipartite_unweighted = stream_louvain(session, BIPARTITE_GRAPH)
    bipartite_weighted   = stream_louvain(session, BIPARTITE_GRAPH, weight_property = "order_count")
    drop_graph(session, BIPARTITE_GRAPH)

candidates = {
    "1. favorite_category":                               latest_df[["customer_id", "favorite_category"]].rename(columns = {"favorite_category": "group"}),
    "2. Louvain, customer-category graph, unweighted":    bipartite_unweighted.rename(columns = {"community_id": "group"}),
    "3. Louvain, customer-category graph, weighted":      bipartite_weighted.rename(columns = {"community_id": "group"}),
    "4. Louvain, customer-similarity graph (community_id)": latest_df[["customer_id", "community_id"]].rename(columns = {"community_id": "group"}),
}

rows = []
for label, grouping in candidates.items():
    m = grouping.merge(segments_df, on = "customer_id")
    rows.append([label, m["group"].nunique(), adjusted_rand_index(m["group"], m["segment"])])

print(tabulate(rows, headers = ["Grouping", "Groups", "ARI vs hidden segments"],
               tablefmt = "simple", floatfmt = ".2f"))

print()
print("community_id vs hidden segment (latest snapshot):")
check = latest_df.merge(segments_df, on = "customer_id")
print(pd.crosstab(check["community_id"], check["segment"]).to_string())

print()
print("favorite_category vs hidden segment (latest snapshot):")
print(pd.crosstab(check["favorite_category"], check["segment"]).to_string())

Grouping                                                Groups    ARI vs hidden segments
----------------------------------------------------  --------  ------------------------
1. favorite_category                                         5                      0.52
2. Louvain, customer-category graph, unweighted              2                      0.08
3. Louvain, customer-category graph, weighted                3                      0.41
4. Louvain, customer-similarity graph (community_id)         5                      0.98

community_id vs hidden segment (latest snapshot):
segment       Books + Sports  Clothing + Books  Clothing + Home  Electronics + Home  Electronics + Sports
community_id                                                                                             
0                          0                 0                0                  23                     0
1                          0                 1               15                   0              

## 11. Does `community_id` Add Predictive Signal?

Recovering hidden segments is nice, but a feature earns its place by helping predictions. We test that directly, without training a model, using only the 30-day-old snapshot:

1. Group customers by the features **as they were 30 days ago**. `community_id` and `favorite_category` use only orders placed up to that date, so there is no leakage.
2. For each group, estimate its mix of product categories from orders up to that date.
3. Score how well that mix predicts the category of every order placed **after** that date. The score is log-loss (the average of -log of the probability given to the category that was actually bought), so lower is better.

The groupings compared are: no grouping at all (everyone shares one category mix), `favorite_category`, `community_id` and the hidden segment as an oracle upper bound that no real feature could beat.

Finally, a bootstrap gives a 95% confidence interval for the improvement of `community_id` over `favorite_category`. It resamples individual held-out orders, so it ignores the fact that orders from one customer are related and the interval is somewhat optimistic.

In [13]:
orders_df = fetch_orders()

snapshot_idx = SNAPSHOT_OFFSETS_DAYS.index(30)
snapshot     = frames[snapshot_idx].merge(segments_df, on = "customer_id")
snapshot["everyone"] = 0

cutoff_ts = int((now - timedelta(days = 30)).timestamp())

groupings = [
    ("No grouping (everyone shares one mix)", "everyone"),
    ("favorite_category",                     "favorite_category"),
    ("community_id",                          "community_id"),
    ("Hidden segment (oracle upper bound)",   "segment"),
]

losses = {label: heldout_losses(orders_df, snapshot, cutoff_ts, col) for label, col in groupings}

n_test = len(next(iter(losses.values())))
rows = [[label, snapshot[col].nunique(), losses[label].mean()] for label, col in groupings]

print(f"Held-out orders (placed after the 30-day snapshot): {n_test}")
print()
print(tabulate(rows, headers = ["Grouping", "Groups", "Held-out log-loss (lower is better)"],
               tablefmt = "simple", floatfmt = ".3f"))

# Paired bootstrap: how much better is community_id than favorite_category?
diff = losses["favorite_category"] - losses["community_id"]
rng  = np.random.default_rng(SEED)
boot = np.array([rng.choice(diff, size = len(diff), replace = True).mean() for _ in range(2000)])
low, high = np.percentile(boot, [2.5, 97.5])

print()
print(f"Improvement of community_id over favorite_category: {diff.mean():.3f} "
      f"(95% CI {low:.3f} to {high:.3f})")

Held-out orders (placed after the 30-day snapshot): 827

Grouping                                 Groups    Held-out log-loss (lower is better)
-------------------------------------  --------  -------------------------------------
No grouping (everyone shares one mix)         1                                  1.565
favorite_category                             5                                  1.171
community_id                                  5                                  1.025
Hidden segment (oracle upper bound)           5                                  1.009

Improvement of community_id over favorite_category: 0.146 (95% CI 0.111 to 0.179)


## 12. Write to Parquet

Feast's offline store is a Parquet file containing every snapshot. From this point on, Feast handles the rest -- it doesn't know or care that some features came from Cypher and one came from GDS.

In [14]:
features_df.to_parquet("feature_repo/data/customer_features.parquet", index = False)

print(f"Wrote {len(features_df)} rows to Parquet.")
print()
print(features_df.groupby("event_timestamp").size().rename("customers").to_string())

Wrote 300 rows to Parquet.

event_timestamp
2026-08-27 18:21:16.895592+00:00    100
2026-09-26 18:21:16.895592+00:00    100
2026-10-26 18:21:16.895592+00:00    100


## 13. Apply Feature Definitions

In [15]:
print("Applying feature definitions...")

result = subprocess.run(
    ["feast", "-c", "feature_repo", "apply"],
    capture_output = True,
    text = True
)

print(result.stdout)

if result.returncode == 0:
    print("Features applied successfully.")
else:
    print("Error:", result.stderr)

Applying feature definitions...
No project found in the repository. Using project name ecommerce_features defined in feature_store.yaml
Applying changes for project ecommerce_features
Created project ecommerce_features
Created entity customer_id
Created feature view customer_features

Created sqlite table ecommerce_features_customer_features


Features applied successfully.


## 14. Materialize Features to the Online Store

Materialization reads the Parquet file and loads the **most recent** value for each customer into the SQLite online store, where it can be retrieved at low latency during inference.

In [16]:
end_date   = datetime.now(timezone.utc)
start_date = end_date - timedelta(days = 90)

print(f"Materializing features from {start_date} to {end_date}...")

result = subprocess.run(
    [
        "feast", "-c", "feature_repo", "materialize",
        start_date.isoformat(),
        end_date.isoformat()
    ],
    capture_output = True,
    text = True
)

print(result.stdout)

if result.returncode == 0:
    print("Features materialized to online store.")
else:
    print("Error:", result.stderr)

Materializing features from 2026-07-28 18:21:21.757788+00:00 to 2026-10-26 18:21:21.757788+00:00...
Materializing 1 feature views from 2026-07-28 18:21:21+00:00 to 2026-10-26 18:21:21+00:00 into the sqlite online store.

customer_features:

Features materialized to online store.


## 15. Online Feature Retrieval

We retrieve features for three customers, one from each activity level: an active customer, a cooling customer and a churned customer. Notice that `community_id` is retrieved alongside the standard features -- from Feast's perspective it's just another column.

> `community_id` is an arbitrary label, so a model should treat it as categorical (for example one-hot or embedding), not as an ordered number.

In [17]:
fs = FeatureStore(repo_path = "feature_repo")

active_customer = next(c for c in latest_df["customer_id"]
                       if c not in churned_customers and c not in cooling_customers)

test_customers = [active_customer, cooling_customers[0], churned_customers[0]]
entity_rows    = [{"customer_id": cid} for cid in test_customers]

feature_vector = fs.get_online_features(
    features = [
        "customer_features:total_orders",
        "customer_features:total_spent",
        "customer_features:avg_order_value",
        "customer_features:days_since_last_order",
        "customer_features:favorite_category",
        "customer_features:community_id",
    ],
    entity_rows = entity_rows
)

retrieved_df = feature_vector.to_df()
print(tabulate(retrieved_df, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

customer_id      days_since_last_order    avg_order_value    total_orders  favorite_category      community_id    total_spent
-------------  -----------------------  -----------------  --------------  -------------------  --------------  -------------
CUST_0081                            5             277.33              28  Books                             4        7765.10
CUST_0007                           35             269.26              26  Electronics                       0        7000.73
CUST_0032                           66             253.08              26  Clothing                          2        6580.14


## 16. Use Features for Prediction

A simple rule-based churn risk function: `days_since_last_order` drives the risk tier and `community_id` is returned alongside so downstream systems can segment or report by community. This demonstrates that the serving interface is the same regardless of how each feature was computed.

> **Staleness:** `days_since_last_order` was computed at materialization time. Served a week later without re-materializing, it would be a week out of date. In production, either re-run materialization on a schedule or store the last-order timestamp and compute recency at request time (for example with a Feast on-demand feature view).

In [18]:
def predict_customer_churn(customer_id):
    """Use features for real-time churn prediction."""

    features = fs.get_online_features(
        features = [
            "customer_features:total_orders",
            "customer_features:total_spent",
            "customer_features:days_since_last_order",
            "customer_features:community_id",
        ],
        entity_rows = [{"customer_id": customer_id}]
    ).to_df()

    days_since = features["days_since_last_order"].iloc[0]
    community  = features["community_id"].iloc[0]

    if days_since > 60:
        risk, action = "HIGH",   "Send 20% discount"
    elif days_since > 30:
        risk, action = "MEDIUM", "Send personalized email"
    else:
        risk, action = "LOW",    "Continue engagement"

    return {
        "customer_id":           customer_id,
        "days_since_last_order": int(days_since),
        "churn_risk":            risk,
        "action":                action,
        "community_id":          int(community),
    }

predictions = [predict_customer_churn(c) for c in test_customers]
print(tabulate(predictions, headers = "keys", tablefmt = "simple"))

customer_id      days_since_last_order  churn_risk    action                     community_id
-------------  -----------------------  ------------  -----------------------  --------------
CUST_0081                            5  LOW           Continue engagement                   4
CUST_0007                           35  MEDIUM        Send personalized email               0
CUST_0032                           66  HIGH          Send 20% discount                     2


## 17. Point-in-Time Historical Features

This is the other half of what a feature store is for: building **training data** in which each row sees only the feature values that were known at that row's timestamp.

We ask Feast for one customer's features at three timestamps. Feast joins each timestamp to the most recent snapshot at or before it, so we should see the 60-day-old snapshot, then the 30-day-old snapshot, then the latest one. Because every snapshot, including `community_id`, was computed using only orders up to its own date, there is no leakage from the future.

In [19]:
target = test_customers[0]

entity_df = pd.DataFrame({
    "customer_id":     [target] * 3,
    "event_timestamp": [
        pd.Timestamp(now - timedelta(days = 45)),
        pd.Timestamp(now - timedelta(days = 20)),
        pd.Timestamp(datetime.now(timezone.utc)),
    ],
})

historical_df = fs.get_historical_features(
    entity_df = entity_df,
    features  = [
        "customer_features:total_orders",
        "customer_features:total_spent",
        "customer_features:avg_order_value",
        "customer_features:days_since_last_order",
        "customer_features:favorite_category",
        "customer_features:community_id",
    ],
).to_df()

print(f"Snapshots in the offline store: "
      f"{', '.join((now - timedelta(days = d)).strftime('%Y-%m-%d') for d in SNAPSHOT_OFFSETS_DAYS)}")
print()
print(tabulate(historical_df, headers = "keys", tablefmt = "simple", showindex = False, floatfmt = ".2f"))

Snapshots in the offline store: 2026-08-27, 2026-09-26, 2026-10-26

customer_id    event_timestamp                     total_orders    total_spent    avg_order_value    days_since_last_order  favorite_category      community_id
-------------  --------------------------------  --------------  -------------  -----------------  -----------------------  -------------------  --------------
CUST_0081      2026-09-11 18:21:16.895592+00:00              12        3609.80             300.82                        2  Books                             4
CUST_0081      2026-10-06 18:21:16.895592+00:00              20        6086.00             304.30                        6  Books                             4
CUST_0081      2026-10-26 18:21:23.949695+00:00              28        7765.10             277.32                        5  Books                             4


## 18. Teardown

In [20]:
with driver.session(database = NEO4J_DATABASE) as session:
    drop_graph(session, BIPARTITE_GRAPH)
    drop_graph(session, SIMILARITY_GRAPH)

driver.close()
print("Driver closed.")

Driver closed.
